# token_analysis_the_verdict
Tokenization analysis for `data/the-verdict.txt` using `llm_from_scratch.utils.token_analysis`.

In [6]:
# Optional Google Drive mount (Colab only)
import os
from pathlib import Path

def in_colab() -> bool:
    return "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ or "google.colab" in str(getattr(__import__("sys"), "modules", {}))

IN_COLAB = in_colab()

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        print("Mounted Google Drive at /content/drive")
    except Exception as e:
        print(f"Could not mount Google Drive: {e}")
else:
    print("Not running in Colab; skipping Drive mount.")


Not running in Colab; skipping Drive mount.


In [7]:
# Resolve project root and paths
import sys

env_root = os.environ.get("LLM_PROJECT_ROOT", "").strip()
default_colab_root = Path("/content/drive/MyDrive/llm-from-scratch-drive")

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists():
            return p
    return start

if env_root:
    PROJECT_ROOT = Path(env_root).expanduser().resolve()
elif default_colab_root.exists():
    PROJECT_ROOT = default_colab_root.resolve()
else:
    PROJECT_ROOT = find_repo_root(Path.cwd().resolve())

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


CWD: /home/markb/llm-from-scratch/notebooks
PROJECT_ROOT: /home/markb/llm-from-scratch
SRC_DIR: /home/markb/llm-from-scratch/src
DATA_DIR: /home/markb/llm-from-scratch/data


In [8]:
# Imports and tokenizer setup
from collections import Counter

import pandas as pd
import tiktoken

from llm_from_scratch.utils import token_analysis as ta

ta.tokenizer = tiktoken.get_encoding("gpt2")
print("Tokenizer ready: gpt2")


Tokenizer ready: gpt2


In [9]:
# Load the verdict text
filenm = "pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_val_abstracts.txt"
text_path = DATA_DIR / filenm
with open(text_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Text path:", text_path)
print("Characters:", len(text))
print("Lines:", text.count("\n") + 1)
print("Preview:\n")
print(text[:500])


Text path: /home/markb/llm-from-scratch/data/pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_val_abstracts.txt
Characters: 2904096
Lines: 1790
Preview:

To better elucidate the socioeconomic and racial differences in women who received postmastectomy radiation therapy with or without a chest wall boost, the records from 4747 women included in the California Cancer Registry were reviewed. Poor and Hispanic women were more likely to receive a chest wall boost than were more affluent and non-Hispanic women. INTRODUCTION: Healthcare disparities in breast cancer treatment have been well documented. We investigated the socioeconomic status (SES) and r


In [14]:
# Count token ids and inspect top tokens
token_id_counts = ta.count_token_ids([text])
top_tokens_df = ta.top_n_token_df(token_id_counts, n=100)

print("Unique token ids:", len(token_id_counts))
print("Total tokens:", sum(token_id_counts.values()))
print(top_tokens_df)


Unique token ids: 15050
Total tokens: 671020
    tokenid  decoded_repr  count  fraction_total
0        13           '.'  23462        0.034965
1        11           ','  19264        0.028709
2        12           '-'  18957        0.028251
3       290        ' and'  15345        0.022868
4       286         ' of'  14821        0.022087
..      ...           ...    ...             ...
95    31155     ' tumors'    822        0.001225
96      407        ' not'    818        0.001219
97     2882   ' response'    808        0.001204
98    10277          'uz'    806        0.001201
99    23005  ' mutations'    802        0.001195

[100 rows x 4 columns]


In [11]:
# Group token counts after left-strip (e.g., ' the' and 'the')
decoded_token_counts = Counter({ta.decode_id(tid): count for tid, count in token_id_counts.items()})
grouped_counts = ta.group_counts_by_lstrip(decoded_token_counts)

grouped_df = pd.DataFrame(grouped_counts.most_common(40), columns=["token_text", "count"])
grouped_df["fraction_total"] = grouped_df["count"] / grouped_df["count"].sum()
grouped_df


,token_text,count,fraction_total
0,.,23673,0.097290
1,",",19275,0.079215
2,-,19090,0.078455
3,and,15507,0.063730
4,of,14909,0.061272
5,the,13824,0.056813
6,in,12686,0.052136
7,(,12142,0.049900
8,to,7035,0.028912
9,with,6850,0.028152


In [12]:
# Build word-like units from GPT token boundaries
units = list(ta.wordlike_units(text))
unit_counts = Counter(units)

print("Total word-like units:", len(units))
print("Unique word-like units:", len(unit_counts))

unit_df = pd.DataFrame(unit_counts.most_common(40), columns=["unit", "count"])
unit_df["fraction_total"] = unit_df["count"] / len(units)
unit_df


Total word-like units: 413377
Unique word-like units: 41442


,unit,count,fraction_total
0,and,15165,0.036686
1,of,14818,0.035846
2,the,13798,0.033379
3,in,10228,0.024743
4,to,6937,0.016781
5,with,6823,0.016506
6,a,4886,0.011820
7,for,3980,0.009628
8,was,3888,0.009405
9,were,3678,0.008897


In [15]:
# Top words using simple split on whitespace, commas, colons, semicolons, and periods
top_simple_words_df = ta.top_n_words_simple_split(text, n=40)
top_simple_words_df


AttributeError: module 'llm_from_scratch.utils.token_analysis' has no attribute 'top_n_words_simple_split'